    

---

# Objectif global:
Préparer un jeu de données consolidé à partir des **données mensuelles de trajets vélo à Chicago (Divvy)** pour une future analyse exploratoire, visualisation ou modélisation dans Power BI.   

**Les étapes réalisées permettent de :**

- Importer les données brutes depuis le web

- Nettoyer et transformer les colonnes

- Calculer des indicateurs utiles (durée, distance, saison, etc.)

- Produire un dataset prêt à analyser ou à visualiser

**NB:** Divvy = le service de vélos en libre-service de la ville de Chicago. Ils rendent publics leurs données de trajets via l'espace de stockage en ligne AWS S3 (Amazon Web Services – Simple Storage Service) qui est un service de stockage de fichiers dans le cloud, proposé par Amazon.

---


# I. Importation et pré-traitement d'un dataset
---



## Importing libraries and loading data
*Le jeu de données à analyser se trouve à l'adresse suivante : (https://divvy-tripdata.s3.amazonaws.com/index.html). Il y a un fichier par mois, pour un total de 12 fichiers. Chaque fichier contient 13 colonnes avec des types de données variés. Nous allons fusionner les fichiers en un seul et le nommer 'combined_data'.*

1. Importation des bibliothèques

In [ ]:
# Importation des bibliothèques
import numpy as np
import pandas as pd
import datetime
import requests
import zipfile
import io
import os
from math import radians, sin, cos, sqrt, atan2

 2. Téléchargement des 12 fichiers mensuels (janv–déc 2021)

In [ ]:
####### Importation des datasets depuis le serveur AWS dédié

# Liste des noms de fichiers, Chaque ZIP contient un CSV correspondant à un mois
file_names = [
    "202101-divvy-tripdata.zip",
    "202102-divvy-tripdata.zip",
    "202103-divvy-tripdata.zip",
    "202104-divvy-tripdata.zip",
    "202105-divvy-tripdata.zip",
    "202106-divvy-tripdata.zip",
    "202107-divvy-tripdata.zip",
    "202108-divvy-tripdata.zip",
    "202109-divvy-tripdata.zip",
    "202110-divvy-tripdata.zip",
    "202111-divvy-tripdata.zip",
    "202112-divvy-tripdata.zip"
]

# URL de base
base_url = "https://divvy-tripdata.s3.amazonaws.com/"

# Télécharger chaque fichier
for file_name in file_names:

    url = base_url + file_name
    response = requests.get(url)

    # Extraire le fichier Zip dans le dossier local "wild_divvy_data"
    with zipfile.ZipFile(io.BytesIO(response.content)) as the_zip:
      the_zip.extractall("wild_divvy_data")


 3. Chargement et concaténation des fichiers CSV

In [ ]:

folder_path = 'wild_divvy_data'

# Obtenir une liste de tous les fichiers CSV dans le dossier local
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Initialiser un DataFrame vide pour stocker les données combinées
combined_data = pd.DataFrame()

# Parcourir chaque fichier CSV grace à une boucle et concaténer toutres ses données dans une DataFrame nommé "combined_data"
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    data = pd.read_csv(file_path)
    combined_data = pd.concat([combined_data, data], ignore_index=True)

# Afficher les données combinées
combined_data

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,89E7AA6C29227EFF,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member
1,0FEFDE2603568365,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual
2,E6159D746B2DBB91,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member
3,B32D3199F1C2E75B,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member
4,83E463F23575F4BF,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5595058,FA66BCAB0D73DDC2,classic_bike,2021-09-22 15:46:57,2021-09-22 16:01:15,Ellis Ave & 83rd St,584,Stony Island Ave & 75th St,KA1503000019,41.744123,-87.599034,41.758670,-87.586883,casual
5595059,1D44DEFB5D36CA04,classic_bike,2021-09-25 16:25:23,2021-09-25 16:40:29,Ellis Ave & 60th St,KA1503000014,Shore Dr & 55th St,TA1308000009,41.785097,-87.601073,41.795212,-87.580715,casual
5595060,6A346EA57FC23C45,classic_bike,2021-09-25 16:26:05,2021-09-25 16:40:30,Ellis Ave & 60th St,KA1503000014,Shore Dr & 55th St,TA1308000009,41.785097,-87.601073,41.795212,-87.580715,casual
5595061,49360AFD771100A6,classic_bike,2021-09-15 17:57:48,2021-09-15 18:24:06,Ellis Ave & 60th St,KA1503000014,Shore Dr & 55th St,TA1308000009,41.785097,-87.601073,41.795212,-87.580715,casual


In [ ]:
# On vérifie le type de données
combined_data.dtypes

ride_id                object
rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id       object
end_station_name       object
end_station_id         object
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
dtype: object

# II. Nettoyage et formatage
---



## Vérification des valeurs manquantes (null) et des doublons (duplicates)

In [ ]:
#Check valeurs manquantes :
nan_count01 = combined_data.isna().sum()
nan_count01

ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name    690809
start_station_id      690806
end_station_name      739170
end_station_id        739170
start_lat                  0
start_lng                  0
end_lat                 4771
end_lng                 4771
member_casual              0
dtype: int64

In [ ]:
# Identifier les lignes dupliquées basées sur l’ensemble des colonnes.
duplicates = combined_data.duplicated()

# Compter le nombre de lignes dupliquée:
dup_count = duplicates.sum()
dup_count

0

*=> Il n’y a aucune ligne dupliquée..*

In [ ]:
# Suppression des lignes avec valeurs manquantes
df01 = combined_data.dropna()
df01.shape

(4588302, 13)

*Nous allons ajouter de nouvelles colonnes au DataFrame. Nous supprimerons les lignes inutiles après avoir ajouté les nouvelles colonnes..*

## 5. Création de nouvelles colonnes
### 5.1 Colonne "distance_travelled_km"

*Nous allons ajouter la distance parcourue (en kilomètres) à partir des coordonnées GPS de départ (latitude, longitude) et d’arrivée.
Pour cela, nous utiliserons la **formule de Haversine**, qui permet de calculer la distance entre deux points sur la surface de la Terre.*

In [ ]:
# Formule de Haversine:
def haversine(lat1, lon1, lat2, lon2):
    # Rayon de la Terre en km :
    R = 6371.0

    # Conversion degrés vers radians (Les fonctions trigonométriques (sin, cos) en Python utilisent les radians, donc on convertit les latitudes/longitudes.)
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # On calcule la différence entre les deux points (latitude et longitude)
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Formule de Haversine:
    ## a => calcule une partie de la distance angulaire entre les deux points.
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2

    ## c => donne l’angle central (en radians) entre les deux points, vu depuis le centre de la Terre.
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    # Distance finale
    distance = R * c

    return distance

In [ ]:
# Copie vers la nouvelle dataframe
df02 = df01.copy(deep=True)

# Calculate distance and store the result in a new column
df02['distance_travelled_km'] = df02.apply(lambda row: haversine(row['start_lat'], row['start_lng'], row['end_lat'], row['end_lng']), axis=1)
df02.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km
0,89E7AA6C29227EFF,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104
1,0FEFDE2603568365,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412
2,E6159D746B2DBB91,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636
3,B32D3199F1C2E75B,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501
4,83E463F23575F4BF,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460


In [ ]:
# verif des données de la colonne 'distance_travelled_km'
df02['distance_travelled_km'].describe()

count    4.588302e+06
mean     2.128845e+00
std      1.879802e+00
min      0.000000e+00
25%      9.038498e-01
50%      1.619725e+00
75%      2.814214e+00
max      3.380018e+01
Name: distance_travelled_km, dtype: float64

*Certaines données présentent une distance nulle.
Il est possible qu’il s’agisse de trajets aller-retour (même station de départ et d’arrivée). Nous allons également ajouter la durée du trajet.*

### 5.2 Colonne "ride_duration_s"

In [ ]:
# Copie vers la nouvelle dataframe
df03 = df02.copy(deep=True)

# Conversion des dates (started_at, ended_at) en format datetime
## On crée une liste contenant les noms des colonnes qui représentent des dates/horaires.
date_columns = ['started_at','ended_at']
## On sélectionne ces deux colonnes dans le DataFrame df03 et On convertit les deux colonnes au format datetime de Pandas
df03[date_columns] = df03[date_columns].apply(pd.to_datetime)

# Calcule de la durée du trajet en secondes et on stocke le résultat dans une nouvelle colonne
df03['ride_duration_s']=(df03['ended_at']-df03['started_at']).dt.total_seconds()
df03.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s
0,89E7AA6C29227EFF,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0
1,0FEFDE2603568365,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0
2,E6159D746B2DBB91,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0
3,B32D3199F1C2E75B,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0
4,83E463F23575F4BF,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0


In [ ]:
# Nous allons supprimer les trajets dont la durée est inférieure à 1 minute.

df03_filtered = df03[df03['ride_duration_s'] >= 60]
df04 =df03_filtered.copy(deep=True)
df04.shape

(4528933, 15)

### 5.3 Colonne "speed_kph"
Nous allons affiner le nettoyage des données en calculant la vitesse en km/h.*

In [ ]:
# calculer la vitesse en km/h :
df04['speed_kph'] = (df04['distance_travelled_km'] / (df04['ride_duration_s'] / 3600))

# Vérifier la répartition des valeurs dans la colonne speed_kph:
df04['speed_kph'].describe()

count    4.528933e+06
mean     9.319658e+00
std      4.740170e+00
min      0.000000e+00
25%      6.635187e+00
50%      9.811789e+00
75%      1.245334e+01
max      6.796607e+01
Name: speed_kph, dtype: float64

*=> Nous allons supprimer les données dont la vitesse dépasse 45 km/h ( la vitesse maximale autorisée pour un vélo aux États-Unis)*

In [ ]:
# copier les données avec une vitesse <45 km/h dans un nouveau DataFrame:
df05 = df04[df04['speed_kph'] <= 45]
df05.shape

(4528930, 16)

## 6. Suppression des données inutiles

*6.1 Supprimer certaines colonnes afin d’accélérer le chargement des données.*

In [ ]:
#Drop columns 'ride_id' and 'speed_kph'
columns_to_drop = ['ride_id', 'speed_kph']
df05_drop = df05.drop(columns=columns_to_drop)
df05_drop.shape

(4528930, 14)

*6.2 Supprimer les espaces superflus (espaces en fin de texte) dans les colonnes de type chaîne de caractères.  
6.3 Certaines données correspondent à des trajets de test (le mot "test" apparaît dans le nom ou l’ID de la station).
Nous allons également supprimer ces lignes de test.*



In [ ]:
# Supprimer les espaces superflus
columns_str = ['rideable_type', 'start_station_name', 'start_station_id',	'end_station_name', 'end_station_id', 'member_casual']
df05_drop[columns_str] = df05_drop[columns_str].astype(str).apply(lambda col: col.str.strip())

#  Filtrer les lignes ne contenant pas "test" dans ID and name, both start and end
columns = ['start_station_name', 'start_station_id',	'end_station_name', 'end_station_id']
wo_test = ~df05_drop[columns].apply(lambda col: col.str.contains('test', case=False)).any(axis=1)

# Garder uniquement les lignes qui ne contiennent pas "test"
df05_filtered = df05_drop[wo_test]
df05_filtered.shape

## 7. Ajout de nouvelles colonnes utiles révéler des tendances basées sur le temps:*

*   day_of_week: Monday, Tuesday, Wednesday and etc.
*   day_type: Weekday or Weekend
*   month: January, February, March and etc.
*   season: Winter, Spring, Summer, Fall





*7.1 Ajout de colonnes 'day_of_week' and 'month'*

In [ ]:
#Add 'day_of_week' and 'month' based on 'started_at'
df_add = df05_filtered.copy(deep=True)
df_add['day_of_week']=df_add['started_at'].dt.strftime('%A')
df_add['month']=df_add['started_at'].dt.strftime('%B')
df_add.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s,day_of_week,month
0,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0,Friday,February
1,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0,Sunday,February
2,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0,Tuesday,February
3,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0,Tuesday,February
4,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0,Tuesday,February


*7.2 Ajout de colonnes 'day_type' : Weekday or Weekend*

In [ ]:
#Fonction pour 'day_type'
def day_cat (d):
  if d == "Saturday" or d == "Sunday":
    return "Weekend"
  else: return "Weekday"

In [ ]:
# Ajouter 'day_type'
df_add['day_type'] = df_add['day_of_week'].apply(day_cat)
df_add.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s,day_of_week,month,day_type
0,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0,Friday,February,Weekday
1,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0,Sunday,February,Weekend
2,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0,Tuesday,February,Weekday
3,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0,Tuesday,February,Weekday
4,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0,Tuesday,February,Weekday


*7.3 Ajout de colonne 'season'*

In [ ]:
# Fonction pour 'season'
def szn (m):
  if m == "December" or m == "January" or m == "February":
    return "Winter"
  elif m == "March" or m == "April" or m == "May":
    return "Spring"
  elif m == "June" or m == "July" or m == "August":
    return "Summer"
  else: return "Fall"

In [ ]:
# Ajout de 'season'
df_add['season'] = df_add['month'].apply(szn)
df_add.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s,day_of_week,month,day_type,season
0,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0,Friday,February,Weekday,Winter
1,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0,Sunday,February,Weekend,Winter
2,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0,Tuesday,February,Weekday,Winter
3,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0,Tuesday,February,Weekday,Winter
4,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0,Tuesday,February,Weekday,Winter


*7.4 Nous allons ajouter une colonne  'route_type' qui indiquera si le trajet est un aller simple ou un aller-retour, en comparant 'start_station_name' and 'end_station_name'*



In [ ]:
#Fonction pour 'route_type'
def rte_ty (row):
  if row['start_station_name'] == row['end_station_name']:
    return "Round trip"
  else: return "One-way trip"

In [ ]:
#Add 'route_type'
df_add['route_type'] = df_add.apply(rte_ty, axis=1)
df_add.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s,day_of_week,month,day_type,season,route_type
0,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0,Friday,February,Weekday,Winter,One-way trip
1,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0,Sunday,February,Weekend,Winter,One-way trip
2,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0,Tuesday,February,Weekday,Winter,One-way trip
3,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0,Tuesday,February,Weekday,Winter,One-way trip
4,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0,Tuesday,February,Weekday,Winter,One-way trip


*7.5 Ajouter la durée du trajet en minutes.*

In [ ]:
#Add 'ride_duration_min'
df_add ['ride_duration_min'] = df_add['ride_duration_s'] / 60
df_add.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,distance_travelled_km,ride_duration_s,day_of_week,month,day_type,season,route_type,ride_duration_min
0,classic_bike,2021-02-12 16:14:56,2021-02-12 16:21:43,Glenwood Ave & Touhy Ave,525,Sheridan Rd & Columbia Ave,660,42.012701,-87.666058,42.004583,-87.661406,member,0.981104,407.0,Friday,February,Weekday,Winter,One-way trip,6.783333
1,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,Glenwood Ave & Touhy Ave,525,Bosworth Ave & Howard St,16806,42.012701,-87.666058,42.019537,-87.669563,casual,0.813412,1171.0,Sunday,February,Weekend,Winter,One-way trip,19.516667
2,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,Clark St & Lake St,KA1503000012,State St & Randolph St,TA1305000029,41.885795,-87.631101,41.884866,-87.627498,member,0.315636,532.0,Tuesday,February,Weekday,Winter,One-way trip,8.866667
3,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,Wood St & Chicago Ave,637,Honore St & Division St,TA1305000034,41.895634,-87.672069,41.903119,-87.673935,member,0.846501,265.0,Tuesday,February,Weekday,Winter,One-way trip,4.416667
4,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,State St & 33rd St,13216,Emerald Ave & 31st St,TA1309000055,41.834733,-87.625827,41.838163,-87.645123,member,1.643460,914.0,Tuesday,February,Weekday,Winter,One-way trip,15.233333


## 8. Renommer certaines colonnes et réorganiser leur ordre afin de structurer le tableau de manière plus lisible.

In [ ]:
#Rename columns and re-arrange columns order
df_format = df_add.copy(deep=True)
df_format.rename(columns = {'rideable_type':'bike_type','started_at':'start_time', 'ended_at':'end_time', 'member_casual':'user_type' }, inplace = True)
new_cols = ['bike_type', 'user_type','start_time','end_time','day_of_week','day_type','month','season','start_station_name','end_station_name','route_type', 'start_lat', 'start_lng', 'end_lat',	'end_lng',	'distance_travelled_km', 'ride_duration_s', 'ride_duration_min']
df_clean=df_format[new_cols]
df_clean.head()

,bike_type,user_type,start_time,end_time,day_of_week,day_type,month,season,start_station_name,end_station_name,route_type,start_lat,start_lng,end_lat,end_lng,distance_travelled_km,ride_duration_s,ride_duration_min
0,classic_bike,member,2021-02-12 16:14:56,2021-02-12 16:21:43,Friday,Weekday,February,Winter,Glenwood Ave & Touhy Ave,Sheridan Rd & Columbia Ave,One-way trip,42.012701,-87.666058,42.004583,-87.661406,0.981104,407.0,6.783333
1,classic_bike,casual,2021-02-14 17:52:38,2021-02-14 18:12:09,Sunday,Weekend,February,Winter,Glenwood Ave & Touhy Ave,Bosworth Ave & Howard St,One-way trip,42.012701,-87.666058,42.019537,-87.669563,0.813412,1171.0,19.516667
2,electric_bike,member,2021-02-09 19:10:18,2021-02-09 19:19:10,Tuesday,Weekday,February,Winter,Clark St & Lake St,State St & Randolph St,One-way trip,41.885795,-87.631101,41.884866,-87.627498,0.315636,532.0,8.866667
3,classic_bike,member,2021-02-02 17:49:41,2021-02-02 17:54:06,Tuesday,Weekday,February,Winter,Wood St & Chicago Ave,Honore St & Division St,One-way trip,41.895634,-87.672069,41.903119,-87.673935,0.846501,265.0,4.416667
4,electric_bike,member,2021-02-23 15:07:23,2021-02-23 15:22:37,Tuesday,Weekday,February,Winter,State St & 33rd St,Emerald Ave & 31st St,One-way trip,41.834733,-87.625827,41.838163,-87.645123,1.643460,914.0,15.233333


In [ ]:
#Check de dataframe finale:
df_clean.shape

(4528312, 18)

## 9.Extraire le DataFrame pour l’analyse

In [ ]:
#Sauvegarde le DataFrame df_clean dans un fichier CSV et Extraction:
from google.colab import files
df_clean.to_csv('cyclistic_clean.csv', index=False)
files.download('cyclistic_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>